In [1]:
# Modified 25-05-15 by PS for testing with ESMValTool processed dataset SAGE-CCI-OMPS+
# See https://github.com/Darwyn72/obs4MIPs/tree/457_PS_SAGE-CCI-OMPS%2B/inputs/ESMValTool 

# Import common libraries
import pandas as pd
import netCDF4 as nc
from netCDF4 import Dataset 
import cmor
import xarray as xr
from xarray.coding.times import encode_cf_datetime
import numpy as np
import cftime
import sys,os,glob

In [2]:
#%% User provided input 
cmorTable = '/Users/paul.smith/demo/Tables/obs4MIPs_Amon.json' ; # Aday,Amon,Lmon,Omon,SImon,fx NOTE: needs full filepath

#EXAMPLEs with command line input - Tried this in Terminal but generates syntax errors 
#1 python -i runCMOR_ESMVAlTool-obs.py o3 SAGE-CCI-OMPS_input.json /Users/paul.smith/demo/Data/OBS6_ESACCI-OZONE_sat_L3_AERmon_o3_198410-202212.nc

#2 python -i runCMOR_ESMVAlTool-obs.py rlut CALIPSO-ICECLOUD_input.json /p/user_pub/PCMDIobs/obs4MIPs_input/ESMValTool/ESACCI-CLOUD/cli_mon_CALIPSO-ICECLOUD-1-00_DLR_200701-201512.nc

# Run in terminal as above to collect inputVarName etc. 
# command_line = True
# if command_line == True:
inputVarName = 'o3'
inputJson = '/Users/paul.smith/demo/Tables/SAGE-CCI-OMPS_input.json'
inputFilePathbgn = '/Users/paul.smith/demo/Data/OBS6_ESACCI-OZONE_sat_L3_AERmon_o3_198410-202212.nc'

###
f = xr.open_dataset(inputFilePathbgn,decode_times=False, decode_cf=False)
# f = f.bounds.add_missing_bounds(axes=['X', 'Y']) # ONLY IF BOUNDS NOT IN INPUT FILE
# f = f.bounds.add_bounds('T') # ONLY IF BOUNDS NOT IN INPUT FILE
d = f[inputVarName].values
vunits = f[inputVarName].units

In [3]:
# print(f.alt16)

In [4]:
### Initialize and run CMOR
cmor.setup(inpath='./',netcdf_file_action=cmor.CMOR_REPLACE_4 ,logfile='cmorLog.' + inputVarName + '.txt')
cmor.dataset_json(inputJson)
cmor.load_table(cmorTable)

cmorLat = cmor.axis("latitude", coord_vals=f.lat[:].values, cell_bounds=f.lat_bnds.values, units="degrees_north")
cmorLon = cmor.axis("longitude", coord_vals=f.lon[:].values, cell_bounds=f.lon_bnds.values, units="degrees_east")
cmorTime = cmor.axis("time", coord_vals=f.time.values, cell_bounds=f.time_bnds.values, units= f.time.units)
axes = [cmorTime, cmorLat, cmorLon]

In [5]:
############ DATASET SPECIFIC
if inputVarName in ['o3']:
    cmorLev = cmor.axis('alt41',coord_vals=f.alt16.values,units = 'km')
    axes = [cmorTime, cmorLev, cmorLat, cmorLon]
############
print(axes)

[2, 3, 0, 1]


In [ ]:
print(inputVarName,vunits,axes,)
# Setup units and create variable to write using cmor - see https://cmor.llnl.gov/mydoc_cmor3_api/#cmor_set_variable_attribute
varid   = cmor.variable(inputVarName,vunits,axes,missing_value=1.e20)

In [ ]:
# Setup units and create variable to write using cmor - see https://cmor.llnl.gov/mydoc_cmor3_api/#cmor_set_variable_attribute
varid   = cmor.variable(inputVarName,vunits,axes,missing_value=1.e20,positive="up")
values  = np.array(d[:],np.float32)
print(values)

In [ ]:
############ DATASET SPECIFIC # Adding o3 for the SAGE-CCI-OMPS+ dataset 
if inputVarName in ['o3']: varid = cmor.variable(inputVarName,vunits,axes,missing_value=1.e20,positive="up")
############
print(varid)

In [ ]:
# Prepare variable for writing, then write and close file - see https://cmor.llnl.gov/mydoc_cmor3_api/#cmor_set_variable_attribute
cmor.set_deflate(varid,1,1,1) ; # shuffle=1,deflate=1,deflate_level=1 - Deflate options compress file data
cmor.write(varid,values) ; # Write variable with time axis
f.close()
cmor.close()
print('done with ',inputVarName)